In [9]:
import sys
sys.path.append('..')

import pandas as pd
from utils.db_utils import write_table

In [10]:
def transform_grad_by_age(file_path):
    df = pd.read_csv(file_path)
    
    # 1. Reshape years into rows (Melt)
    df_long = df.melt(
        id_vars=["statistics"],
        var_name="year",
        value_name="value"
    )
    
    # 2. Extract Age Group and Category
    def parse_stats(stat_name):
        if "_" in stat_name and any(char.isdigit() or char in "<>≥" for char in stat_name):
            parts = stat_name.split("_", 1)
            return parts[0].strip(), parts[1]
        else:
            return "Total", stat_name

    # Apply the splitting logic
    df_long[['age_group', 'category']] = df_long['statistics'].apply(
        lambda x: pd.Series(parse_stats(x))
    )
    
    # 3. Pivot the categories into columns
    # Now we index by Year AND Age Group
    df_pivot = df_long.pivot_table(
        index=["year", "age_group"],
        columns="category",
        values="value",
        aggfunc="first"
    ).reset_index()
    
    # 4. Clean up formatting
    df_pivot.columns.name = None
    
    # Convert numeric columns to float
    cols_to_fix = [c for c in df_pivot.columns if c not in ["year", "age_group"]]
    for col in cols_to_fix:
        df_pivot[col] = pd.to_numeric(df_pivot[col], errors='coerce')
        
    df_pivot = df_pivot.sort_values(["year"]).reset_index(drop=True)

    num_cols = [col for col in df_pivot.columns if col not in ["year", "age_group"]]

    for col in num_cols:
        df_pivot[col] = (
            df_pivot[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .astype(float) * 1000
    )
    
    return df_pivot

In [11]:
df_final = transform_grad_by_age("../../data/Total Graduates by Age Group.csv")
df_final.head(10)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xa1 in position 37: invalid start byte